### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [3]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    # TODO: при ошибке конвертации amount использовать raise ... from ...
    arr = raw.split('|')
    try:
        arr[-1] = int(arr[-1])
    except ValueError as e:
        raise TransferAmountError('количество должно быть числом') from e
    if arr[-1] <= 0:
        raise TransferAmountError('количество должно быть больше 0') from e
    return {'transfer_id':arr[0], 'from_user':arr[1], 'to_user':arr[2], 'amount':arr[3]}



def apply_transfer(transfer, wallets):
    # TODO: проверить существование аккаунтов
    # TODO: запретить перевод самому себе
    # TODO: проверить совпадение валют
    # TODO: проверить баланс отправителя
    # TODO: обновить балансы и вернуть dict результата
    if not transfer['from_user'] in wallets or not transfer['to_user'] in wallets:
        raise AccountNotFoundError('аккаунт(-ы) не найдены') 
    if transfer['from_user'] == transfer['to_user']:
        raise AccountNotFoundError('нельзя перевести самому себе')
    if wallets[transfer['from_user']]['currency'] != wallets[transfer['to_user']]['currency']:
        raise CurrencyMismatchError('несовпадение валют')
    if wallets[transfer['from_user']]['balance'] < transfer['amount']:
        raise InsufficientFundsError('на счету недостаточно средств')
    wallets[transfer['from_user']]['balance'] -= transfer['amount']
    return {'from_user': transfer['from_user'], 'isGood': True}
    
    


    


def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []
    for el in rows:
        try:
            prs = parse_transfer(el)
            successes.append(apply_transfer(prs, wallets))
        except Exception as e:
            errors.append((el, type(e).__name__, e))
    return (successes, errors)




# TODO: вызвать process_batch(rows, wallets)
# TODO: вывести успешные переводы
# TODO: вывести ошибки по типам
# TODO: вывести итоговые балансы
# TODO: найти richest_usd_user
res = process_batch(rows, wallets)
for el in res[0]:
    print(f'{el} - Успешный')

type_errors = {}
for el in res[1]:
    type_errors.setdefault(el[1], set())
    type_errors[el[1]].add(el)

print(type_errors)

m = 0
user = None
print('Итоговые балансы:')
for key, val in wallets.items():
    print(f'{key} - {val['balance']}')
    if val['currency'] == 'USD' and val['balance'] > m:
        m =val['balance']
        user = key

print(f'Самый богатый USD-пользователь: {user}')



{'from_user': 'alice', 'isGood': True} - Успешный
{'from_user': 'bob', 'isGood': True} - Успешный
{'InsufficientFundsError': {('TR-101|bob|dave|700', 'InsufficientFundsError', InsufficientFundsError('на счету недостаточно средств'))}, 'CurrencyMismatchError': {('TR-102|alice|carol|50', 'CurrencyMismatchError', CurrencyMismatchError('несовпадение валют'))}, 'AccountNotFoundError': {('TR-103|eve|bob|30', 'AccountNotFoundError', AccountNotFoundError('аккаунт(-ы) не найдены')), ('TR-104|dave|dave|10', 'AccountNotFoundError', AccountNotFoundError('нельзя перевести самому себе'))}, 'TransferAmountError': {('TR-105|bob|alice|abc', 'TransferAmountError', TransferAmountError('количество должно быть числом'))}}
Итоговые балансы:
alice - 1000.0
bob - 350.0
carol - 900.0
dave - 150.0
Самый богатый USD-пользователь: alice
